# Automated Bird Species Recognition with Auxiliary Concept Bottlenecks
## Google Colab GPU Multi-Task Training Pipeline (CSE 4385)

**Project:** Calibrated Concept-Based Feedback Model for Fine-Grained Bird Recognition  
**Dataset:** Caltech-UCSD Birds-200-2011 (CUB-200-2011) — 11,788 images, 200 species, 112 concepts  
**Primary Objective:** 200 Bird Species Recognition (`[batch, 200]`)  
**Auxiliary Objective:** 112 Visual Concept Prediction (`[batch, 112]`)  

---
### Pipeline Steps:
1. **Hardware Verification:** Checks GPU availability (T4/V100/A100) and halts on CPU.
2. **Google Drive Integration:** Interactive Drive mounting and project folder setup.
3. **High-Speed Staging:** Stages dataset from Drive archive to local SSD (`/content/data/`) to prevent I/O throttling.
4. **Integrity Audit:** Verifies 0% partition leakage across Train (4,794), Val (600), Calib (600), Test (5,794).
5. **Fast GPU Smoke Test:** Runs a 2-batch verification on the selected architecture.
6. **Resilient Training:** Executes multi-task AdamW + AMP training with automatic Drive checkpoint mirroring and seamless resume.

## Step 1: Compute Hardware & GPU Diagnostics
Verifies CUDA GPU availability and inspects available VRAM.

In [ ]:
import os
import sys
import torch

# Check CUDA
if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL: No GPU detected!\n"
        "Please enable GPU acceleration before proceeding:\n"
        "  Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
    )

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[SUCCESS] GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
print(f"PyTorch Version: {torch.__version__} | CUDA: {torch.version.cuda}")

## Step 2: Google Drive Mounting & Project Initialization
Mounts Google Drive interactively and creates the persistent folder hierarchy under `MyDrive/CalibratedBirdRecognition/`.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CalibratedBirdRecognition')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[SUCCESS] Google Drive mounted. Project root: {DRIVE_ROOT}")

## Step 3: Clone or Link Research Codebase
Navigates to the workspace directory and verifies dependencies.

In [ ]:
%cd /content
# Install essential requirements
!pip install -q pyyaml scikit-learn pillow

import base64
import io
import os
import sys
import zipfile
from pathlib import Path

# 1. Ensure /content is on Python sys.path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 2. Self-Extract Embedded Codebase if src is missing
if not (Path('/content/src').exists() and Path('/content/outputs/dataset_manifest.json').exists()):
    print('[SETUP] Auto-unpacking embedded research codebase (src, configs, outputs, data)...')
    _EMBEDDED_ZIP_B64 = b'''...'''
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(_EMBEDDED_ZIP_B64))) as _z:
        _z.extractall('/content')
    print('[PASS] Research codebase successfully extracted to /content!')


## Step 4: High-Speed Dataset Staging
Extracts dataset from Drive into `/content/data/CUB_200_2011/` on the local NVMe SSD to avoid Google Drive FUSE latency.

In [ ]:
from src.cloud.dataset_sync import DatasetStagingManager

staging_mgr = DatasetStagingManager(
    drive_source_dir=DRIVE_ROOT / 'dataset/CUB_200_2011',
    local_staging_dir='/content/data/CUB_200_2011',
)
staging_res = staging_mgr.stage_dataset()
print(f"Dataset Staging Status: {staging_res['status']}")

## Step 5: Dataset & Partition Integrity Verification
Confirms 0% partition leakage across all splits and verifies 112 canonical concepts.

In [ ]:
!python scripts/verify_preprocessed_dataset.py --dataset-root /content/data/CUB_200_2011

## Step 6: GPU Smoke Test (2 Batches)
Runs an isolated forward+backward smoke test on the chosen model without full training.

In [ ]:
# Smoke test ResNet-50 dual-head model on GPU
!python scripts/run_experiment.py --model resnet50 --smoke-test --device cuda --use-amp

## Step 7: Launch Full Training with Automatic Drive Recovery
Executes full multi-task training with automatic Drive checkpoint mirroring.
Supported models:
- `resnet50` (E2: ResNet-50 Dual-Head)
- `resnet50_cbam` (E3: ResNet-50 + CBAM Attention)
- `resnet50_cbam_fpn` (E4: ResNet-50 + CBAM + FPN Multiscale)
- `efficientnet_b2` (E5: EfficientNet-B2 Dual-Head)
- `convnext_tiny` (E6: ConvNeXt-Tiny Dual-Head)

*To resume an interrupted run, add `--resume`.*

In [ ]:
# Example: Train ResNet-50 with auto-resume enabled
!python scripts/run_experiment.py \
    --model resnet50 \
    --epochs 40 \
    --batch-size 32 \
    --lambda-concept 0.2 \
    --device cuda \
    --use-amp \
    --resume